# Module 13 — Iterators and generators

`yield` is the thing in this course with no equivalent in C and only a distant one in
Java. It is also the last piece of the `for` loop you have been using since module 03,
because a `for` loop was never about lists.

## 1. What a `for` loop actually does

Three steps, and none of them mentions a list:

1. call `iter(thing)` to get an **iterator**;
2. call `next()` on it, over and over;
3. stop when it raises `StopIteration`.

That is the whole protocol, and it is why `for` works on a list, a string, a dict, a
file and a `Log` of your own from module 12.

In [ ]:
values = [21.7, 91.0]
iterator = iter(values)

print(type(iterator).__name__)
print(next(iterator))
print(next(iterator))

try:
    next(iterator)
except StopIteration:
    print("StopIteration -- which is what ends a for loop, silently")

Two things worth separating, because the words get used interchangeably and mean
different things:

- An **iterable** is something you can call `iter()` on: a list, a string, a dict, a
  range.
- An **iterator** is what you get back: something with `__next__`, which is used up as
  you go.

An iterator is also an iterable — `iter(iterator)` returns the same object — which is
what lets you pass one to a `for` loop. The reverse does not hold: a list is not an
iterator, which is why you can loop over the same list twice.

In [ ]:
values = [21.7, 91.0]

print(list(values), list(values))  # a list can be walked again

iterator = iter(values)
print(list(iterator), list(iterator))  # an iterator cannot -- the second is empty

print(iter(iterator) is iterator)  # an iterator returns itself from iter()

Writing one by hand is the version Java makes you write: a class with `__iter__` and
`__next__`, and the state carried in attributes.

In [ ]:
class Countdown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self  # I am my own iterator

    def __next__(self):
        if self.current <= 0:
            raise StopIteration  # this is how the loop learns to stop
        self.current -= 1
        return self.current + 1


print(list(Countdown(3)))
print([n for n in Countdown(2)])

That is `Iterator<T>` with `hasNext()` folded into the exception. It works, and after
the next section you will not write it again.

## 2. `yield`

A function containing `yield` is a **generator function**. Calling it runs **none** of
the body: it builds a generator object and hands it back. The body runs a piece at a
time, each `next()` continuing from where the last `yield` left off.

In [ ]:
def count_to(limit):
    print("   [body starts]")
    for number in range(limit):  # noqa: UP028 -- `yield from` is section 5; this is the plain form
        yield number
    print("   [body ends]")


generator = count_to(3)
print("called, and nothing has run:", type(generator).__name__)

print("first next():", next(generator))
print("second next():", next(generator))
print("the rest:", list(generator))

Read the order of the printed lines. `[body starts]` did not appear until the first
`next()`, and `[body ends]` appeared when the loop inside the generator finished — at
which point the generator raised `StopIteration` and `list()` stopped.

The state is the paused function: local variables, the position in the loop, the call
stack of that frame. Nothing had to be moved into attributes. That is what the
`Countdown` class above was doing by hand.

In [ ]:
def readings():
    yield 21.7
    yield 91.0


values = readings()
first = list(values)

# The generator has been walked once. What does a second walk give?
assert first == ...
assert list(values) == ...

**A generator is used up.** The second `list()` is empty, and nothing raises — which
is the failure mode to watch for. A function that takes an iterable and walks it twice
works on a list and quietly returns nothing on the second pass when handed a
generator.

If you need it twice, either keep a list (`values = list(generator)`) or call the
generator function again to get a fresh one.

## 3. Why this matters: size

A list comprehension builds the whole result. A generator builds one item at a time
and forgets it.

In [ ]:
import sys

as_list = [n * n for n in range(100_000)]
as_generator = (n * n for n in range(100_000))  # round brackets: a generator expression

print(f"list:      {sys.getsizeof(as_list):>8} bytes")
print(f"generator: {sys.getsizeof(as_generator):>8} bytes")
print(sum(as_generator))  # and it still adds up to the same thing

The generator's size does not depend on `100_000` — there is nothing in it but the
paused function. That is the whole argument, and it is why module 08 could read a 40 GB
file line by line: a file object is an iterator over its own lines.

The rule of thumb:

- **A generator** when the sequence is large, infinite, expensive per item, or you
  will stop early.
- **A list** when you need it more than once, need its length, need to index it, or it
  is small enough that the question is not interesting.

`sum`, `max`, `any`, `all`, `sorted` and `"".join` all take an iterable, so the
brackets can simply be dropped: `sum(n * n for n in values)`.

## 4. Generators over a file

The shape you will actually write: a generator that reads, filters and converts, one
line at a time, with the whole pipeline still lazy.

In [ ]:
import io

RAW = "tag;value\nTH-01;21.7\nTH-04;n/a\nTH-09;23.1\n"


def rows(text):
    """One dict per data line, and nothing held but the current line."""
    lines = iter(io.StringIO(text))  # stands in for open(path) -- both are iterators
    header = next(lines).rstrip("\n").split(";")
    for line in lines:
        yield dict(zip(header, line.rstrip("\n").split(";")))


def readings(rows_):
    for row in rows_:
        try:
            yield float(row["value"])
        except ValueError:
            continue  # module 09: skip what cannot be converted


pipeline = readings(rows(RAW))
print(type(pipeline).__name__)
print(list(pipeline))

Nothing was read until `list()` asked. Each stage pulls one item from the stage before
it, so memory stays flat however long the input is — and the same three functions work
on a four-line string and on a file that does not fit in memory.

`zip` here is another lazy one. So are `enumerate`, `map`, `filter` and `reversed`:
they all hand back iterators rather than lists.

In [ ]:
print(type(zip([1], [2])).__name__, type(enumerate([1])).__name__, type(map(str, [1])).__name__)
print(type(range(3)).__name__)  # range is NOT an iterator -- it is a lazy sequence

numbers = range(3)
print(list(numbers), list(numbers))  # so it can be walked twice
print(iter(numbers) is numbers)  # while an iterator would return itself

## 5. `yield from`

Yielding everything out of another iterable is a loop; `yield from` is the same thing
in one line, and it is what you want when a generator delegates to another.

`ruff` reports the loop version as `UP028`, "replace `yield` over `for` loop with
`yield from`", and offers to rewrite it. Take that as the recommendation it is; the
`# noqa` below only keeps the comparison readable.

In [ ]:
def with_loop():
    yield "start"
    for value in ["a", "b"]:  # noqa: UP028 -- ruff says use `yield from`, which is the point
        yield value
    yield "end"


def with_yield_from():
    yield "start"
    yield from ["a", "b"]
    yield "end"


print(list(with_loop()) == list(with_yield_from()))
print(list(with_yield_from()))

In [ ]:
def files(names):
    for name in names:
        yield from lines_of(name)  # each file's lines, flattened into one stream


def lines_of(name):
    yield f"{name}:1"
    yield f"{name}:2"


print(list(files(["a.log", "b.log"])))

## 6. `return` inside a generator

A `return` ends the generator. The value goes into the `StopIteration`, which is where
`yield from` picks it up — and which is why a plain loop over the generator never sees
it.

In [ ]:
def read_with_count():
    yield 21.7
    yield 91.0
    return 2  # not yielded: this is the generator's result


generator = read_with_count()
print(list(generator))  # the return value is nowhere in here

generator = read_with_count()
next(generator)
next(generator)
try:
    next(generator)
except StopIteration as stop:
    print("returned:", stop.value)

In practice: **do not put information in a generator's return value** unless you are
writing something that consumes it with `yield from`. A caller doing the obvious thing
will never see it. If a count has to come back, yield it, or hand the caller an object
that has both.

## 7. `itertools`

The standard library's iterator toolbox. Four that earn their place early:

In [ ]:
import itertools

print(list(itertools.islice(itertools.count(10), 3)))  # count is infinite; islice takes 3
print(list(itertools.chain([1, 2], [3])))  # one stream out of several
print(list(itertools.pairwise([1, 2, 3])))  # consecutive pairs -- differences, gaps
print([(key, list(group)) for key, group in itertools.groupby([1, 1, 2])])

`groupby` groups **consecutive** equal items, not all equal items — sort first if you
meant the other thing. That is the one in this list that surprises people, and it is
the same design as the Unix `uniq`.

`itertools.count` is infinite, and so are generators you write with `while True`. That
is not a problem as long as something downstream stops: `islice`, a `break`, or a
condition in the consumer.

In [ ]:
def rising():
    value = 0
    while True:  # never ends on its own
        yield value
        value += 1


for number in rising():
    if number > 3:
        break
    print(number)

## 8. Where Java lands

`Iterator<T>` and `Iterable<T>` are the same protocol with different spelling —
`hasNext()` plus `next()` where Python has `next()` plus `StopIteration`.

What Java has no equivalent for is `yield`: writing an iterator there means a class
with fields for the state, which is the `Countdown` in section 1. Java streams are
lazy in the same way as a generator pipeline, and their `map`/`filter` correspond
closely to a generator expression — but a stream is built from combinators rather than
by writing an ordinary function with a `yield` in it, and it is also single-use, which
is the same trap as section 2.

---

`exercises/` is next: `exercise_01.py` to `exercise_06.py`, `exercise_09.py`, and two
in `thinking.md` with nothing to run.

That is the end of Part 3. Part 4 is tools: decorators — where the `@` finally gets
explained — `pytest`, HTTP, scraping, pandas and SQL.